# Explore the orders and back up three findings

*Data Analysis in Python · Lesson 8 final exercise*

The head of e-commerce wants three findings from the store's orders that they can act on: which orders are unusual, whether some product categories come back more than others, and which order metrics move together. Each one needs a number behind it, and two of them go in one figure for the leadership meeting.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the parameters cell and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Read `ORDERS_FILE`, `ORDER_ITEMS_FILE` and `PRODUCTS_FILE` into the `orders`, `order_items` and `products` tables, and profile them with lesson 2.'s questions: rows, columns, types, missing values, and what one row is.
2. Create the `valid_orders` table: the rows of the `orders` table whose `status` isn't `Cancelled`. Every step after this one uses valid orders only.
3. Unusual orders: create `upper_fence`, the IQR rule's upper fence of the `valid_orders` table's `subtotal`, the order value before shipping and tax, and the `outlier_orders` table: the valid orders above it.
4. Returns: create the `item_rows` table: the `order_items` table with each line's product `category` from the `products` table and its order's `status` from the `orders` table, without the lines of cancelled orders. Then create `return_rates`: for each category, the share of its lines whose order was returned, as a Series of fractions.
5. Relationships: create the `correlations` table: the correlations of the `valid_orders` table's `num_items`, `subtotal`, `shipping` and `total`. Then create `strongest_pair`, a tuple of the two column names with the strongest correlation, positive or negative, between two metrics where neither is calculated from the other (`total` is `subtotal` plus `shipping` plus `tax`), and `strongest_r`, their correlation.
6. The figure: create `findings_fig`, a Figure with two charts side by side and a headline, and name them:
   - `rate_ax`: the left Axes, `return_rates` as horizontal bars with the highest rate on top, its value axis starting at 0 and its ticks formatted as percentages
   - `value_ax`: the right Axes, a histogram of the `valid_orders` table's `subtotal`, with a vertical line at `upper_fence`

   Save `findings_fig` to `FIGURE_FILE`.
7. Add a text cell for the head of e-commerce with three findings, each with the number that supports it: what the unusual orders are and how much revenue they bring, whether return rates differ by category, and which order metrics move together, and why.
8. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then open the PNG from the Files panel.

**Expected output**: The `orders` table has 65,106 rows, and the `valid_orders` table 61,732. `upper_fence` is `214.99`, and the `outlier_orders` table has 2,246 orders, worth $604,781.19, 11.1% of the valid orders' $5,447,144.17. The `item_rows` table has 87,910 rows. `return_rates` has 12 categories, from Outerwear's 12.0% to Sweaters' 13.6%. The `correlations` table is 4 × 4, `strongest_pair` is `num_items` and `subtotal`, and `strongest_r` is about `0.7955`. `findings_fig` has two Axes, `rate_ax` with the 12 rates as sorted horizontal bars from 0, and `value_ax` with a histogram of the 61,732 valid orders' subtotals and a line at the fence, under a headline. `FIGURE_FILE`, `orders_findings.png`, is saved and isn't blank.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
ORDERS_FILE = DATA_URL + "orders.csv"
ORDER_ITEMS_FILE = DATA_URL + "order_items.csv"
PRODUCTS_FILE = DATA_URL + "products.csv"
FIGURE_FILE = "orders_findings.png"

In [ ]:
orders = pd.read_csv(ORDERS_FILE)
order_items = pd.read_csv(ORDER_ITEMS_FILE)
products = pd.read_csv(PRODUCTS_FILE)
orders.shape, order_items.shape, products.shape

In [ ]:
orders.info()
orders["status"].value_counts()

In [ ]:
valid_orders = orders[orders["status"] != "Cancelled"]
q1, q3 = valid_orders["subtotal"].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
outlier_orders = valid_orders[valid_orders["subtotal"] > upper_fence]
len(valid_orders), upper_fence, len(outlier_orders), outlier_orders["subtotal"].sum() / valid_orders["subtotal"].sum()

In [ ]:
outlier_orders["num_items"].mean(), valid_orders["num_items"].mean()

In [ ]:
item_rows = order_items.merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
item_rows = item_rows.merge(orders[["order_id", "status"]], on="order_id", validate="many_to_one")
item_rows = item_rows[item_rows["status"] != "Cancelled"]
return_rates = (item_rows["status"] == "Returned").groupby(item_rows["category"]).mean()
len(item_rows), return_rates.sort_values()

In [ ]:
correlations = valid_orders[["num_items", "subtotal", "shipping", "total"]].corr()
strongest_pair = ("num_items", "subtotal")
strongest_r = correlations.loc["num_items", "subtotal"]
correlations.round(4)

In [ ]:
findings_fig, (rate_ax, value_ax) = plt.subplots(1, 2, figsize=(13, 4.5))
return_rates.sort_values().plot.barh(ax=rate_ax)
rate_ax.set_xlim(left=0)
rate_ax.xaxis.set_major_formatter("{x:.0%}")
rate_ax.bar_label(rate_ax.containers[0], fmt="{:.1%}", padding=3)
rate_ax.set_title("Share of order lines returned, by category")
rate_ax.set_ylabel("")
value_ax.hist(valid_orders["subtotal"], bins=50)
value_ax.axvline(upper_fence, color="red", linestyle="--", label=f"Outlier fence: ${upper_fence:,.2f}")
value_ax.xaxis.set_major_formatter("${x:,.0f}")
value_ax.set_title("Order value (subtotal), valid orders")
value_ax.set_ylabel("Orders")
value_ax.legend()
findings_fig.suptitle("Returns are about 12–14% in every category; orders above $215 are 3.6% of orders and 11% of revenue", fontsize=13, x=0.01, ha="left")
findings_fig.tight_layout()
findings_fig.savefig(FIGURE_FILE, dpi=150, bbox_inches="tight")
plt.show()

**For the head of e-commerce.** 61,732 orders in 2024 and 2025 that weren't cancelled, worth $5,447,144.17 before shipping and tax.

- **Unusual orders are big baskets, not errors.** 2,246 orders, 3.6%, are above $214.99, the IQR rule's fence. They bring $604,781.19, 11.1% of revenue, and have 3.44 items on average, against 1.54 for all orders. Keep them: they're the best customers' orders.
- **Return rates barely differ by category.** From Outerwear's 12.0% to Sweaters' 13.6% of order lines, around an average of 12.5%. That's 1.6 percentage points: no category is a returns problem on its own, and a chart that makes Sweaters look like one has a cut-off axis.
- **Bigger baskets mean bigger orders, and free shipping.** The number of items and the subtotal correlate at 0.80. Subtotal and shipping correlate at −0.74, because every order of $75 or more ships free. Subtotal and total correlate at 0.9992, but that's a formula: total is subtotal plus shipping plus tax.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your charts, your file and the cells that have run. It doesn't change anything.
def check_your_work():
    import os
    import sys
    import matplotlib.figure
    import matplotlib.pyplot as plt
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def bars(ax):
        """Return the bars' values from bottom to top, or left to right, and whether they're horizontal."""
        patches = [patch for patch in ax.patches if patch.get_width() and patch.get_height()]
        if not patches:
            return [], None
        horizontal = len({round(patch.get_height(), 6) for patch in patches}) == 1 and len({round(patch.get_width(), 6) for patch in patches}) > 1
        if horizontal:
            ordered = sorted(patches, key=lambda patch: patch.get_y())
            ordered = ordered[::-1] if ax.yaxis_inverted() else ordered
            return [patch.get_width() for patch in ordered], True
        ordered = sorted(patches, key=lambda patch: patch.get_x())
        ordered = ordered[::-1] if ax.xaxis_inverted() else ordered
        return [patch.get_height() for patch in ordered], False

    orders = found.get("orders")
    valid = found.get("valid_orders")
    upper = found.get("upper_fence")
    outliers = found.get("outlier_orders")
    if not isinstance(orders, pd.DataFrame) or len(orders) != 65106:
        results.append((False, "orders should be the orders file, 65,106 rows: pd.read_csv(ORDERS_FILE)."))
    elif not isinstance(valid, pd.DataFrame):
        results.append((False, "valid_orders should be the orders table's rows that aren't cancelled: orders[orders[\"status\"] != \"Cancelled\"]."))
    elif len(valid) == 65106:
        results.append((False, "The valid_orders table has all 65,106 orders: the 3,374 cancelled ones are still in it. Leave out the rows whose status is Cancelled."))
    elif len(valid) != 61732:
        results.append((False, f"The valid_orders table should have 61,732 rows, every order that isn't cancelled. It has {len(valid):,}."))
    elif not number(upper):
        results.append((False, f"upper_fence should be a number: the IQR rule's upper fence of the valid orders' subtotal. It is {upper!r}."))
    elif abs(upper - 219.615) < 0.001:
        results.append((False, "upper_fence is 219.62: that's total, which includes shipping and tax. The order value in the brief is subtotal."))
    elif abs(upper - 214.99) > 0.001:
        results.append((False, f"upper_fence should be q3 + 1.5 * (q3 - q1) of the valid orders' subtotal: 214.99. It is {upper:,.3f}."))
    elif not isinstance(outliers, pd.DataFrame) or len(outliers) != 2246:
        results.append((False, f"outlier_orders should be the valid orders whose subtotal is above upper_fence: 2,246 rows. It has {len(outliers) if isinstance(outliers, pd.DataFrame) else outliers!r}."))
    else:
        share = outliers["subtotal"].sum() / valid["subtotal"].sum()
        results.append((True, f"2,246 orders above ${upper:,.2f}, worth ${outliers['subtotal'].sum():,.2f}: {share:.1%} of the valid orders' revenue, with {outliers['num_items'].mean():.2f} items each against {valid['num_items'].mean():.2f}."))

    rows = found.get("item_rows")
    rates = found.get("return_rates")
    if isinstance(rates, pd.DataFrame) and rates.shape[1] == 1:
        rates = rates.iloc[:, 0]
    if not isinstance(rows, pd.DataFrame) or not {"category", "status"} <= set(rows.columns):
        results.append((False, "item_rows should be the order_items table with category from products and status from orders: two merge()s, on product_id and on order_id."))
    elif len(rows) == 92723:
        results.append((False, "The item_rows table has 92,723 rows: the lines of cancelled orders are still in it. A cancelled order can't be returned, so leave them out."))
    elif len(rows) != 87910:
        results.append((False, f"The item_rows table should have 87,910 rows: every order line, without the cancelled orders' lines. It has {len(rows):,}. A merge that adds rows means a key repeats: use validate=\"many_to_one\"."))
    elif not isinstance(rates, pd.Series) or "Sweaters" not in rates.index:
        if isinstance(rates, pd.Series) and "Women" in rates.index:
            results.append((False, "return_rates is by department. The brief asks for each product category: group by category."))
        else:
            results.append((False, "return_rates should be a Series with one rate per category: (item_rows[\"status\"] == \"Returned\").groupby(item_rows[\"category\"]).mean()."))
    elif abs(rates["Sweaters"] - 0.1292) < 0.0001:
        results.append((False, "return_rates' Sweaters is 12.9%: the cancelled orders' lines are in the denominator. Leave them out of item_rows."))
    elif abs(rates["Sweaters"] - 13.62) < 0.01:
        results.append((False, "return_rates is in percent: Sweaters is 13.62. Keep rates as fractions, and format them on the chart."))
    elif len(rates) != 12 or abs(rates["Sweaters"] - 0.1362) > 0.0001 or abs(rates["Outerwear"] - 0.1199) > 0.0001:
        results.append((False, f"return_rates should have 12 categories, from Outerwear's 0.1199 to Sweaters' 0.1362: the share of each category's lines whose order was returned. It has {len(rates)}, with Sweaters at {rates['Sweaters']:.4f}."))
    else:
        results.append((True, f"Return rates by category run from {rates.min():.1%} ({rates.idxmin()}) to {rates.max():.1%} ({rates.idxmax()}): {(rates.max() - rates.min()) * 100:.1f} percentage points apart."))

    correlations = found.get("correlations")
    pair = found.get("strongest_pair")
    r = found.get("strongest_r")
    if not isinstance(correlations, pd.DataFrame) or correlations.shape != (4, 4):
        results.append((False, "correlations should be a 4 × 4 table: valid_orders[[\"num_items\", \"subtotal\", \"shipping\", \"total\"]].corr()."))
    elif abs(correlations.loc["num_items", "subtotal"] - 0.795563) < 0.000001:
        results.append((False, "The correlations table includes the cancelled orders. Correlate the valid_orders table's columns."))
    elif not isinstance(pair, (tuple, list)) or len(pair) != 2:
        results.append((False, f"strongest_pair should be a tuple of two column names, such as (\"shipping\", \"total\"). It is {pair!r}."))
    elif set(pair) == {"subtotal", "total"}:
        results.append((False, "strongest_pair is subtotal and total, 0.9992: total is subtotal plus shipping plus tax, so of course they move together. That's a formula, not a finding. Leave out pairs where one is calculated from the other."))
    elif len(set(pair)) == 1:
        results.append((False, f"strongest_pair is {pair[0]} with itself: the diagonal is always 1. Pick two different columns."))
    elif set(pair) == {"subtotal", "shipping"}:
        results.append((False, "strongest_pair is subtotal and shipping, -0.7417: strong, but num_items and subtotal are stronger, 0.7955. Compare correlations by size, whatever their sign."))
    elif set(pair) != {"num_items", "subtotal"}:
        results.append((False, f"strongest_pair should be the two columns with the strongest correlation, positive or negative, where neither is calculated from the other. It is {tuple(pair)}."))
    elif not number(r) or abs(r - 0.7955) > 0.0001:
        results.append((False, f"strongest_r should be num_items' and subtotal's correlation, about 0.7955. It is {r!r}."))
    else:
        results.append((True, f"num_items and subtotal correlate at {r:.4f}, the strongest pair that isn't a formula. Subtotal and shipping, {correlations.loc['subtotal', 'shipping']:.4f}, is the store's free shipping from $75."))

    fig = found.get("findings_fig")
    rate_ax, value_ax = found.get("rate_ax"), found.get("value_ax")
    if not isinstance(fig, matplotlib.figure.Figure):
        results.append((False, f"findings_fig should be the Figure: findings_fig, (rate_ax, value_ax) = plt.subplots(1, 2, figsize=(13, 4.5)). It is a {type(fig).__name__}."))
    elif len(fig.axes) != 2 or rate_ax not in fig.axes or value_ax not in fig.axes:
        results.append((False, f"findings_fig should have two Axes, rate_ax and value_ax, from one plt.subplots(1, 2). It has {len(fig.axes)}."))
    else:
        values, horizontal = bars(rate_ax)
        counts = [patch.get_height() for patch in value_ax.patches if patch.get_width()]
        fence_lines = [line for line in value_ax.get_lines() if len(set(line.get_xdata())) == 1 and number(line.get_xdata()[0]) and abs(line.get_xdata()[0] - 214.99) < 0.01]
        if len(values) != 12 or abs(max(values) - 0.1362) > 0.0001:
            results.append((False, f"rate_ax should have 12 bars, return_rates' values, with Sweaters' 0.1362 the longest. It has {len(values)}."))
        elif horizontal is False:
            results.append((False, "rate_ax's bars are vertical. Make them horizontal, so the category names fit: plot.barh(ax=rate_ax)."))
        elif values != sorted(values):
            results.append((False, "rate_ax's bars should be sorted with the highest rate on top: return_rates.sort_values().plot.barh(ax=rate_ax)."))
        elif rate_ax.get_xlim()[0] != 0:
            start = rate_ax.get_xlim()[0]
            results.append((False, f"rate_ax's value axis starts at {start:.1%}, so Sweaters' bar is {(max(values) - start) / (min(values) - start):.1f} times Outerwear's, for rates {(max(values) - min(values)) * 100:.1f} points apart. Start it at 0: rate_ax.set_xlim(left=0)."))
        elif "%" not in "".join(label.get_text() for label in rate_ax.get_xticklabels()):
            results.append((False, "rate_ax's ticks aren't percentages. Format them: rate_ax.xaxis.set_major_formatter(\"{x:.0%}\")."))
        elif sum(counts) == 65106:
            results.append((False, "value_ax's histogram counts 65,106 orders: the cancelled ones too. Draw the valid_orders table's subtotal."))
        elif sum(counts) != 61732:
            results.append((False, f"value_ax should be a histogram of the 61,732 valid orders' subtotal: value_ax.hist(valid_orders[\"subtotal\"], bins=50). Its bars count {sum(counts):,.0f}."))
        elif not fence_lines:
            results.append((False, "value_ax has no vertical line at upper_fence. Add one: value_ax.axvline(upper_fence, color=\"red\", linestyle=\"--\")."))
        elif not fig.get_suptitle().strip():
            results.append((False, "findings_fig has no headline. Add one that states a finding: findings_fig.suptitle(\"…\")."))
        else:
            results.append((True, f"findings_fig has the 12 return rates, sorted from 0, and the 61,732 order values with the fence, under the headline {fig.get_suptitle()!r}."))

    figure_file = found.get("FIGURE_FILE", "orders_findings.png")
    saved = isinstance(history, list) and any("savefig" in source for source in history)
    if not os.path.exists(figure_file) or not saved:
        results.append((False, f"{figure_file} hasn't been saved in this session. Save the Figure: findings_fig.savefig(FIGURE_FILE, dpi=150, bbox_inches=\"tight\")."))
    elif plt.imread(figure_file)[..., :3].std() == 0:
        results.append((False, f"{figure_file} is blank. plt.savefig() in a later cell saves a new, empty Figure. Save it by name: findings_fig.savefig(FIGURE_FILE)."))
    else:
        results.append((True, f"{figure_file} is saved, and it isn't blank. Open it from the Files panel before you send it."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the head of e-commerce will read.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: three new tables</b></summary>

Lesson 2.: read each file, then ask each table how many rows it has, which columns and types, which values are missing, and what one row is. The tables share `order_id` and `product_id`.

~~~python
orders = pd.read_csv(...)
order_items = pd.read_csv(...)
products = pd.read_csv(...)
orders.info()
orders["status"].value_counts()
~~~

</details>

<details>
<summary><b>Step 2: valid orders</b></summary>

Lesson 2.: a filter on `status`. A cancelled order never shipped, so it can be neither revenue nor a return.

~~~python
valid_orders = orders[orders[...] != ...]
len(valid_orders)
~~~

</details>

<details>
<summary><b>Step 3: unusual orders</b></summary>

Topic 8.2.: the quartiles of the valid orders' `subtotal`, then the fence 1.5 IQRs above the third one. Look at what the outliers have in common before you call them anything.

~~~python
q1, q3 = valid_orders[...].quantile([0.25, 0.75])
upper_fence = ...
outlier_orders = valid_orders[...]
len(outlier_orders), outlier_orders["subtotal"].sum() / valid_orders["subtotal"].sum()
~~~

</details>

<details>
<summary><b>Step 4: returns by category</b></summary>

Lesson 5.: two joins, each many order lines to one product or one order, so say so with `validate`. Then the share of `True` in a column of `True` and `False` is its mean.

~~~python
item_rows = order_items.merge(products[["product_id", "category"]], on=..., validate="many_to_one")
item_rows = item_rows.merge(orders[["order_id", "status"]], on=..., validate="many_to_one")
item_rows = item_rows[item_rows["status"] != ...]
return_rates = (item_rows["status"] == "Returned").groupby(item_rows[...]).mean()
return_rates.sort_values()
~~~

</details>

<details>
<summary><b>Step 5: the strongest relationship</b></summary>

Topic 8.3.: `corr()` on four columns. A correlation near 1 between two columns usually means one is calculated from the other: leave those pairs out, and compare the rest by size, whatever their sign.

~~~python
correlations = valid_orders[[...]].corr()
correlations.round(4)
strongest_pair = (..., ...)
strongest_r = correlations.loc[..., ...]
~~~

</details>

<details>
<summary><b>Step 6: the figure</b></summary>

Topics 8.4. and 8.5.: two Axes from one `plt.subplots(1, 2)`, sorted bars with the axis from 0 and percentage ticks, a histogram with `axvline()`, a headline with `suptitle()`, and `savefig()` on the Figure, not on `plt`.

~~~python
findings_fig, (rate_ax, value_ax) = plt.subplots(1, 2, figsize=(13, 4.5))
return_rates.sort_values().plot.barh(ax=rate_ax)
rate_ax.set_xlim(left=0)
rate_ax.xaxis.set_major_formatter(...)
value_ax.hist(..., bins=50)
value_ax.axvline(...)
findings_fig.suptitle(...)
findings_fig.tight_layout()
findings_fig.savefig(FIGURE_FILE, dpi=150, bbox_inches="tight")
~~~

</details>

<details>
<summary><b>Step 7: the text cell</b></summary>

Choose **Insert → Text cell**. One finding per bullet, each with its number. For the outliers, compare their items per order with the valid orders'. For returns, say how far apart the highest and lowest rates are, not only which is highest. For the relationship, look at shipping against subtotal: what does the store charge for shipping above $75?

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
orders = pd.read_csv(ORDERS_FILE)
order_items = pd.read_csv(ORDER_ITEMS_FILE)
products = pd.read_csv(PRODUCTS_FILE)
orders.shape, order_items.shape, products.shape
```

```python
orders.info()
orders["status"].value_counts()
```

```python
valid_orders = orders[orders["status"] != "Cancelled"]
q1, q3 = valid_orders["subtotal"].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
outlier_orders = valid_orders[valid_orders["subtotal"] > upper_fence]
len(valid_orders), upper_fence, len(outlier_orders), outlier_orders["subtotal"].sum() / valid_orders["subtotal"].sum()
```

```python
outlier_orders["num_items"].mean(), valid_orders["num_items"].mean()
```

```python
item_rows = order_items.merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
item_rows = item_rows.merge(orders[["order_id", "status"]], on="order_id", validate="many_to_one")
item_rows = item_rows[item_rows["status"] != "Cancelled"]
return_rates = (item_rows["status"] == "Returned").groupby(item_rows["category"]).mean()
len(item_rows), return_rates.sort_values()
```

```python
correlations = valid_orders[["num_items", "subtotal", "shipping", "total"]].corr()
strongest_pair = ("num_items", "subtotal")
strongest_r = correlations.loc["num_items", "subtotal"]
correlations.round(4)
```

```python
findings_fig, (rate_ax, value_ax) = plt.subplots(1, 2, figsize=(13, 4.5))
return_rates.sort_values().plot.barh(ax=rate_ax)
rate_ax.set_xlim(left=0)
rate_ax.xaxis.set_major_formatter("{x:.0%}")
rate_ax.bar_label(rate_ax.containers[0], fmt="{:.1%}", padding=3)
rate_ax.set_title("Share of order lines returned, by category")
rate_ax.set_ylabel("")
value_ax.hist(valid_orders["subtotal"], bins=50)
value_ax.axvline(upper_fence, color="red", linestyle="--", label=f"Outlier fence: ${upper_fence:,.2f}")
value_ax.xaxis.set_major_formatter("${x:,.0f}")
value_ax.set_title("Order value (subtotal), valid orders")
value_ax.set_ylabel("Orders")
value_ax.legend()
findings_fig.suptitle("Returns are about 12–14% in every category; orders above $215 are 3.6% of orders and 11% of revenue", fontsize=13, x=0.01, ha="left")
findings_fig.tight_layout()
findings_fig.savefig(FIGURE_FILE, dpi=150, bbox_inches="tight")
plt.show()
```

```markdown
**For the head of e-commerce.** 61,732 orders in 2024 and 2025 that weren't cancelled, worth $5,447,144.17 before shipping and tax.

- **Unusual orders are big baskets, not errors.** 2,246 orders, 3.6%, are above $214.99, the IQR rule's fence. They bring $604,781.19, 11.1% of revenue, and have 3.44 items on average, against 1.54 for all orders. Keep them: they're the best customers' orders.
- **Return rates barely differ by category.** From Outerwear's 12.0% to Sweaters' 13.6% of order lines, around an average of 12.5%. That's 1.6 percentage points: no category is a returns problem on its own, and a chart that makes Sweaters look like one has a cut-off axis.
- **Bigger baskets mean bigger orders, and free shipping.** The number of items and the subtotal correlate at 0.80. Subtotal and shipping correlate at −0.74, because every order of $75 or more ships free. Subtotal and total correlate at 0.9992, but that's a formula: total is subtotal plus shipping plus tax.
```

</details>